# Olist E-Commerce Data Cleaning & Transformation

## Objective

Transform the raw Olist e-commerce datasets into clean, consistent, and analytics-ready datasets.

The transformations in this phase are based on issues identified during data profiling. Raw source data will remain unchanged, and cleaned datasets will be written separately to `data/processed/`.

### Transformation Goals

- Standardize column names and correct inconsistent naming.
- Convert date and timestamp fields to appropriate datetime types.
- Handle missing values based on their business meaning.
- Standardize categorical fields where necessary.
- Translate product categories from Portuguese to English.
- Create useful derived fields for later analysis.
- Validate transformed data before loading it into SQL.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)

dataset_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# Reading with pandas, storing in datasets dictionary with name as key
datasets = {}

for name, filename in dataset_files.items():
    file_path = RAW_DATA_DIR / filename
    dataframe = pd.read_csv(file_path)
    datasets[name] = dataframe

    
for name, df in datasets.items():
    print(f"{name:<22} {df.shape}")

Raw data: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\raw
Processed data: d:\Documents\Career\Projects\retail-business-intelligence-platform\data\processed
customers              (99441, 5)
geolocation            (1000163, 5)
order_items            (112650, 7)
payments               (103886, 5)
reviews                (99224, 7)
orders                 (99441, 8)
products               (32951, 9)
sellers                (3095, 4)
category_translation   (71, 2)


In [3]:
datasets["products"] = datasets["products"].rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    }
)
print(datasets["products"].columns.tolist())

['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


## Datetime Standardization

Convert date and timestamp columns from strings into pandas datetime values so they can be used for time-based calculations, filtering, aggregation, and downstream SQL/Power BI analysis.

In [4]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for dataset_name, columns in datetime_columns.items():

    for column in columns:
        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column]
        )

for dataset_name, columns in datetime_columns.items():

    print(f"\n{dataset_name.upper()}")

    for column in columns:
        print(
            column,
            "->",
            datasets[dataset_name][column].dtype
        )


ORDERS
order_purchase_timestamp -> datetime64[us]
order_approved_at -> datetime64[us]
order_delivered_carrier_date -> datetime64[us]
order_delivered_customer_date -> datetime64[us]
order_estimated_delivery_date -> datetime64[us]

ORDER_ITEMS
shipping_limit_date -> datetime64[us]

REVIEWS
review_creation_date -> datetime64[us]
review_answer_timestamp -> datetime64[us]


## Missing-Value Handling

Handle missing values according to their business meaning and intended analytical use. Missing records are not automatically removed, since some null values represent optional information or incomplete lifecycle events rather than unusable data.

### Review Text

In [5]:
# Fill missing review text with descriptive values
datasets["reviews"]["review_comment_title"] = (datasets["reviews"]["review_comment_title"].fillna("No title"))

datasets["reviews"]["review_comment_message"] = (datasets["reviews"]["review_comment_message"].fillna("No comment"))

print(datasets["reviews"][["review_comment_title", "review_comment_message"]].isna().sum())

print(datasets["reviews"][["review_comment_title", "review_comment_message"]].head())

review_comment_title      0
review_comment_message    0
dtype: int64
  review_comment_title                             review_comment_message
0             No title                                         No comment
1             No title                                         No comment
2             No title                                         No comment
3             No title              Recebi bem antes do prazo estipulado.
4             No title  Parabéns lojas lannister adorei comprar pela I...


Missing review titles were replaced with `"No title"`, while missing review messages were replaced with `"No comment"`.

The fields were handled separately because a customer may provide one without providing the other. This preserves the distinction between an absent title and an absent written comment while retaining the review record and its numeric score.

### Products

In [6]:
datasets["products"]["product_category_name"] = (datasets["products"]["product_category_name"].fillna("unknown"))
print("Missing product categories:",datasets["products"]["product_category_name"].isna().sum())
print(datasets["products"][datasets["products"]["product_category_name"]=="unknown"].head())

Missing product categories: 0
                           product_id product_category_name  \
105  a41e356c76fab66334f36de622ecbd3a               unknown   
128  d8dee61c2034d6d075997acef1870e9b               unknown   
145  56139431d72cd51f19eb9f7dae4d1617               unknown   
154  46b48281eb6d663ced748f324108c733               unknown   
197  5fb61f482620cb672f5e586bb132eae9               unknown   

     product_name_length  product_description_length  product_photos_qty  \
105                  NaN                         NaN                 NaN   
128                  NaN                         NaN                 NaN   
145                  NaN                         NaN                 NaN   
154                  NaN                         NaN                 NaN   
197                  NaN                         NaN                 NaN   

     product_weight_g  product_length_cm  product_height_cm  product_width_cm  
105             650.0               17.0              

We'll leave these numeric metadata fields `NULL`: `product_name_length`, `product_description_length`, `product_photos_qty`, `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` but lets add a flag that will help us filter these missing values later.

In [ ]:
datasets["products"]["product_metadata_missing"] = (
    datasets["products"][
        [
            "product_name_length",
            "product_description_length",
            "product_photos_qty"
        ]
    ]
    .isna()
    .all(axis=1)
)

measurement_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

datasets["products"]["product_measurements_missing"] = (
    datasets["products"][measurement_columns]
    .isna().all(axis=1) # checks whether every selected column is missing (NaN) for each row.
)

print(
    datasets["products"]["product_metadata_missing"].value_counts()
)

print(
    datasets["products"]["product_measurements_missing"].value_counts()
)

product_metadata_missing
False    32341
True       610
Name: count, dtype: int64
product_measurements_missing
False    32949
True         2
Name: count, dtype: int64
